In [ ]:
# ============================================
# CSE-4155 Introduction to Machine Learning
# Lab 03 - Classification, Perceptron,
#          Non-linear Feature Mapping,
#          and Image Data Augmentation
# ============================================

import os
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from PIL import (
    Image,
    ImageEnhance,
    ImageFilter,
    ImageOps
)

# For reproducible experiments
np.random.seed(42)


In [ ]:
# ============================================
# 1. GENERATE A BINARY CLASSIFICATION DATASET
# ============================================

def generate_classification_data(
    n_per_class=50,
    seed=42
):
    """
    Generate a simple 2D linearly-separable binary dataset.

    Class -1:
        centered around (-2, -2)

    Class +1:
        centered around (+2, +2)

    Returns:
        X : shape (m, 2)
        y : labels in {-1, +1}
    """

    rng = np.random.default_rng(seed)

    class_negative = rng.normal(
        loc=[-2.0, -2.0],
        scale=[0.8, 0.8],
        size=(n_per_class, 2)
    )

    class_positive = rng.normal(
        loc=[2.0, 2.0],
        scale=[0.8, 0.8],
        size=(n_per_class, 2)
    )

    X = np.vstack((
        class_negative,
        class_positive
    ))

    y = np.hstack((
        -np.ones(n_per_class),
        np.ones(n_per_class)
    ))

    return X, y


In [ ]:
X_linear, y_linear = generate_classification_data()

print("Number of samples:", len(y_linear))
print("Number of features:", X_linear.shape[1])

print("\nFirst 5 feature vectors:")
print(X_linear[:5])

print("\nFirst 5 labels:")
print(y_linear[:5])


In [ ]:
# ============================================
# 2. PLOT THE CLASSIFICATION DATA
# ============================================

def plot_binary_data(
    X,
    y,
    title="Binary Classification Data"
):
    """
    Plot a two-dimensional binary dataset.
    """

    plt.figure(figsize=(8, 6))

    negative = (y == -1)
    positive = (y == 1)

    plt.scatter(
        X[negative, 0],
        X[negative, 1],
        marker="o",
        label="Class -1"
    )

    plt.scatter(
        X[positive, 0],
        X[positive, 1],
        marker="x",
        label="Class +1"
    )

    plt.xlabel("x1")
    plt.ylabel("x2")
    plt.title(title)

    plt.legend()
    plt.grid(alpha=0.3)
    plt.tight_layout()
    plt.show()


In [ ]:
plot_binary_data(
    X_linear,
    y_linear,
    title="Linearly Separable Binary Classification Data"
)


In [ ]:
# ============================================
# 3. TARGET LABEL ENCODING
# ============================================

def encode_zero_one_to_perceptron(y):
    """
    Convert binary labels:

        0 -> -1
        1 -> +1

    This is convenient for the perceptron because
    correctness can be checked using:

        y_i * (theta^T x_i) > 0
    """

    y = np.asarray(y)

    encoded = np.where(
        y == 0,
        -1,
        1
    )

    return encoded


def decode_perceptron_to_zero_one(y):
    """
    Convert:

        -1 -> 0
        +1 -> 1
    """

    y = np.asarray(y)

    decoded = np.where(
        y == -1,
        0,
        1
    )

    return decoded


In [ ]:
# Small demonstration

y_zero_one = np.array([0, 1, 1, 0, 1])

y_encoded = encode_zero_one_to_perceptron(
    y_zero_one
)

print("Original labels:", y_zero_one)
print("Perceptron labels:", y_encoded)
print(
    "Decoded labels:",
    decode_perceptron_to_zero_one(y_encoded)
)


In [ ]:
# ============================================
# 4. ADD CONSTANT / DUMMY FEATURE
# ============================================

def add_constant_feature(X):
    """
    Lift each data point by adding x0 = 1.

    Original 2D point:
        (x1, x2)

    Lifted 3D point:
        (1, x1, x2)

    Then an affine decision boundary

        theta_0 + theta_1*x1 + theta_2*x2 = 0

    can be written as

        theta^T x = 0

    in the lifted feature space.
    """

    X = np.asarray(X, dtype=float)

    X_lifted = np.column_stack((
        np.ones(len(X)),
        X
    ))

    return X_lifted


In [ ]:
X_linear_lifted = add_constant_feature(
    X_linear
)

print("Original shape:", X_linear.shape)
print("Lifted shape:", X_linear_lifted.shape)

print("\nOriginal first point:")
print(X_linear[0])

print("\nLifted first point:")
print(X_linear_lifted[0])


In [ ]:
# ============================================
# 5. PERCEPTRON HYPOTHESIS / PREDICTION
# ============================================

def perceptron_score(X, theta):
    """
    Compute the linear score:

        z = X theta
    """

    return X.dot(theta)


def perceptron_predict(X, theta):
    """
    Predict:

        +1, if theta^T x >= 0
        -1, otherwise
    """

    scores = perceptron_score(
        X,
        theta
    )

    predictions = np.where(
        scores >= 0,
        1,
        -1
    )

    return predictions


In [ ]:
# ============================================
# 6. PERCEPTRON LOSS
# ============================================

def perceptron_loss(X, y, theta):
    """
    Per-sample perceptron loss:

        L_i = max(0, -y_i * theta^T x_i)

    Average loss is returned for convenient monitoring.

    If y_i and the score have the same sign,
    the sample is correctly classified and
    contributes zero loss.
    """

    scores = X.dot(theta)

    losses = np.maximum(
        0,
        -y * scores
    )

    return np.mean(losses)


In [ ]:
# ============================================
# 7. PERCEPTRON TRAINING ALGORITHM
# ============================================

def train_perceptron(
    X,
    y,
    learning_rate=1.0,
    epochs=50,
    seed=42
):
    """
    Train the perceptron from scratch.

    Update condition:
        y_i * (theta^T x_i) <= 0

    Update rule:
        theta <- theta + eta * y_i * x_i

    Returns:
        theta
        mistakes_history
        loss_history
    """

    X = np.asarray(X, dtype=float)
    y = np.asarray(y, dtype=float)

    n_features = X.shape[1]

    # Initialize all parameters with zero
    theta = np.zeros(n_features)

    mistakes_history = []
    loss_history = []

    rng = np.random.default_rng(seed)

    for epoch in range(epochs):

        # Shuffle sample order each epoch
        indices = rng.permutation(len(y))

        mistakes = 0

        for i in indices:

            score = X[i].dot(theta)

            # Misclassified or exactly on boundary
            if y[i] * score <= 0:

                theta = (
                    theta
                    + learning_rate
                    * y[i]
                    * X[i]
                )

                mistakes += 1

        current_loss = perceptron_loss(
            X,
            y,
            theta
        )

        mistakes_history.append(
            mistakes
        )

        loss_history.append(
            current_loss
        )

        # For linearly separable data,
        # zero mistakes means convergence.
        if mistakes == 0:
            break

    return (
        theta,
        mistakes_history,
        loss_history
    )


In [ ]:
# ============================================
# 8. TRAIN ON LINEARLY-SEPARABLE DATA
# ============================================

(
    theta_linear,
    mistakes_linear,
    loss_linear
) = train_perceptron(
    X_linear_lifted,
    y_linear,
    learning_rate=1.0,
    epochs=100,
    seed=42
)

print("Training completed.")

print("\nLearned parameters:")
for j, value in enumerate(theta_linear):
    print(f"theta_{j} = {value:.6f}")

print("\nNumber of epochs used:")
print(len(mistakes_linear))

print("\nMistakes in final epoch:")
print(mistakes_linear[-1])


In [ ]:
# ============================================
# 9. EVALUATE CLASSIFICATION
# ============================================

def classification_accuracy(
    X,
    y,
    theta
):
    """
    Compute simple classification accuracy.
    """

    predictions = perceptron_predict(
        X,
        theta
    )

    accuracy = np.mean(
        predictions == y
    )

    return predictions, accuracy


In [ ]:
predictions_linear, accuracy_linear = (
    classification_accuracy(
        X_linear_lifted,
        y_linear,
        theta_linear
    )
)

print(
    f"Training Accuracy = "
    f"{accuracy_linear * 100:.2f}%"
)


In [ ]:
# ============================================
# 10. PLOT MISTAKES / LOSS AGAINST EPOCH
# ============================================

epochs_used = np.arange(
    1,
    len(mistakes_linear) + 1
)

plt.figure(figsize=(8, 5))

plt.plot(
    epochs_used,
    mistakes_linear,
    marker="o"
)

plt.xlabel("Epoch")
plt.ylabel("Number of Mistakes")
plt.title("Perceptron Mistakes During Training")

plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()


plt.figure(figsize=(8, 5))

plt.plot(
    epochs_used,
    loss_linear
)

plt.xlabel("Epoch")
plt.ylabel("Average Perceptron Loss")
plt.title("Perceptron Loss During Training")

plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()


In [ ]:
# ============================================
# 11. PLOT LEARNED DECISION BOUNDARY
# ============================================

def plot_decision_boundary(
    X_original,
    y,
    theta,
    title="Perceptron Decision Boundary"
):
    """
    For:

        theta_0
        + theta_1*x1
        + theta_2*x2 = 0

    solve for x2:

        x2 =
        -(theta_0 + theta_1*x1) / theta_2
    """

    plt.figure(figsize=(8, 6))

    negative = (y == -1)
    positive = (y == 1)

    plt.scatter(
        X_original[negative, 0],
        X_original[negative, 1],
        marker="o",
        label="Class -1"
    )

    plt.scatter(
        X_original[positive, 0],
        X_original[positive, 1],
        marker="x",
        label="Class +1"
    )

    x1_values = np.linspace(
        X_original[:, 0].min() - 1,
        X_original[:, 0].max() + 1,
        200
    )

    if abs(theta[2]) > 1e-12:

        x2_values = -(
            theta[0]
            + theta[1] * x1_values
        ) / theta[2]

        plt.plot(
            x1_values,
            x2_values,
            linewidth=2,
            label="Decision Boundary"
        )

    plt.xlabel("x1")
    plt.ylabel("x2")
    plt.title(title)

    plt.legend()
    plt.grid(alpha=0.3)
    plt.tight_layout()
    plt.show()


In [ ]:
plot_decision_boundary(
    X_linear,
    y_linear,
    theta_linear,
    title="Learned Linear Decision Boundary"
)


In [ ]:
# ============================================
# 12. RELATION WITH LINEAR REGRESSION
# ============================================

def fit_linear_regression_closed_form(
    X,
    y
):
    """
    Demonstration only.

    A linear regression model can fit labels
    -1 and +1 as continuous targets and then
    threshold the output at zero.

    This is possible, but squared-error regression
    is not designed specifically for classification.
    """

    theta = np.linalg.pinv(X).dot(y)

    return theta


In [ ]:
theta_regression = (
    fit_linear_regression_closed_form(
        X_linear_lifted,
        y_linear
    )
)

predictions_regression = np.where(
    X_linear_lifted.dot(
        theta_regression
    ) >= 0,
    1,
    -1
)

accuracy_regression = np.mean(
    predictions_regression
    == y_linear
)

print("Linear-regression parameters:")
print(theta_regression)

print(
    "\nClassification accuracy after "
    "thresholding regression output:"
)

print(
    f"{accuracy_regression * 100:.2f}%"
)

print(
    "\nObservation:"
)

print(
    "Linear regression produces a continuous "
    "score and can be thresholded, but its "
    "squared-error objective is different from "
    "a classification-specific learning rule."
)


In [ ]:
# ============================================
# 13. GENERATE NON-LINEAR CIRCULAR DATA
# ============================================

def generate_circular_data(
    n=300,
    seed=42
):
    """
    Generate data whose true boundary is circular.

    Class +1:
        points inside radius approximately 1.6

    Class -1:
        points outside that radius

    No single straight line can perfectly
    separate these classes in the original
    (x1, x2) space.
    """

    rng = np.random.default_rng(seed)

    X = rng.uniform(
        -3,
        3,
        size=(n, 2)
    )

    radius_squared = np.sum(
        X ** 2,
        axis=1
    )

    y = np.where(
        radius_squared <= 1.6 ** 2,
        1,
        -1
    )

    return X, y


In [ ]:
X_nonlinear, y_nonlinear = (
    generate_circular_data()
)

plot_binary_data(
    X_nonlinear,
    y_nonlinear,
    title="Non-linear Classification Dataset"
)


In [ ]:
# ============================================
# 14. FEATURE MAPPING FOR NON-LINEAR BOUNDARY
# ============================================

def circular_feature_map(X):
    """
    Map:

        (x1, x2)

    to:

        (1, x1^2, x2^2)

    In this new feature space, a linear equation

        theta_0
        + theta_1*x1^2
        + theta_2*x2^2 = 0

    corresponds to a curved boundary in the
    original input space.
    """

    X = np.asarray(X, dtype=float)

    X_mapped = np.column_stack((
        np.ones(len(X)),
        X[:, 0] ** 2,
        X[:, 1] ** 2
    ))

    return X_mapped


In [ ]:
X_nonlinear_mapped = (
    circular_feature_map(
        X_nonlinear
    )
)

(
    theta_nonlinear,
    mistakes_nonlinear,
    loss_nonlinear
) = train_perceptron(
    X_nonlinear_mapped,
    y_nonlinear,
    learning_rate=0.1,
    epochs=200,
    seed=42
)

predictions_nonlinear, accuracy_nonlinear = (
    classification_accuracy(
        X_nonlinear_mapped,
        y_nonlinear,
        theta_nonlinear
    )
)

print("Mapped-feature parameters:")
print(theta_nonlinear)

print(
    "\nMapped-feature training accuracy = "
    f"{accuracy_nonlinear * 100:.2f}%"
)

print(
    "\nObservation:"
)

print(
    "The perceptron is still linear in the "
    "transformed features, but the resulting "
    "decision boundary is non-linear in the "
    "original x1-x2 plane."
)


In [ ]:
# ============================================
# 15. PLOT NON-LINEAR DECISION BOUNDARY
# ============================================

def plot_nonlinear_boundary(
    X,
    y,
    theta
):
    """
    Plot the zero contour of:

        theta_0
        + theta_1*x1^2
        + theta_2*x2^2 = 0
    """

    plt.figure(figsize=(8, 6))

    negative = (y == -1)
    positive = (y == 1)

    plt.scatter(
        X[negative, 0],
        X[negative, 1],
        marker="o",
        label="Class -1"
    )

    plt.scatter(
        X[positive, 0],
        X[positive, 1],
        marker="x",
        label="Class +1"
    )

    x1_grid = np.linspace(
        X[:, 0].min() - 0.3,
        X[:, 0].max() + 0.3,
        250
    )

    x2_grid = np.linspace(
        X[:, 1].min() - 0.3,
        X[:, 1].max() + 0.3,
        250
    )

    X1, X2 = np.meshgrid(
        x1_grid,
        x2_grid
    )

    Z = (
        theta[0]
        + theta[1] * X1 ** 2
        + theta[2] * X2 ** 2
    )

    plt.contour(
        X1,
        X2,
        Z,
        levels=[0],
        linewidths=2
    )

    plt.xlabel("x1")
    plt.ylabel("x2")
    plt.title(
        "Non-linear Boundary from "
        "Linear Perceptron in Mapped Space"
    )

    plt.legend()
    plt.grid(alpha=0.3)
    plt.tight_layout()
    plt.show()


In [ ]:
plot_nonlinear_boundary(
    X_nonlinear,
    y_nonlinear,
    theta_nonlinear
)


In [ ]:
# ============================================
# 16. OFFICIAL LAB TASK - IMAGE DATASET
# ============================================

# Suggested sample dataset:
#
# Recyclable Waste Image Dataset
# Mendeley Data
# DOI: 10.17632/h5pxbsdz4m.1
#
# It contains 1,000 author-collected images
# related to recyclable waste categories.
#
# Download/extract the dataset manually first,
# then set DATASET_ROOT below to the folder
# containing the original images.

DATASET_ROOT = Path(
    "recyclable_waste_dataset"
)

AUGMENTED_ROOT = Path(
    "recyclable_waste_augmented"
)

print("Expected original dataset folder:")
print(DATASET_ROOT.resolve())

print("\nAugmented output folder:")
print(AUGMENTED_ROOT.resolve())


In [ ]:
# ============================================
# 17. FIND IMAGE FILES
# ============================================

IMAGE_EXTENSIONS = {
    ".jpg",
    ".jpeg",
    ".png",
    ".bmp",
    ".webp"
}


def find_images(root):
    """
    Recursively find image files.
    """

    root = Path(root)

    if not root.exists():
        return []

    image_paths = [
        path
        for path in root.rglob("*")
        if (
            path.is_file()
            and path.suffix.lower()
            in IMAGE_EXTENSIONS
        )
    ]

    return sorted(image_paths)


In [ ]:
original_image_paths = (
    find_images(
        DATASET_ROOT
    )
)

print(
    "Number of original images found:",
    len(original_image_paths)
)

if len(original_image_paths) == 0:

    print(
        "\nNo images were found."
    )

    print(
        "Download/extract the Mendeley dataset "
        "and update DATASET_ROOT, then rerun "
        "this section."
    )


In [ ]:
# ============================================
# 18. IMAGE AUGMENTATION FUNCTIONS
# ============================================

def center_zoom(
    image,
    zoom_factor=1.15
):
    """
    Zoom into the center and resize back
    to the original size.
    """

    width, height = image.size

    crop_width = int(
        width / zoom_factor
    )

    crop_height = int(
        height / zoom_factor
    )

    left = (
        width - crop_width
    ) // 2

    top = (
        height - crop_height
    ) // 2

    right = left + crop_width
    bottom = top + crop_height

    cropped = image.crop((
        left,
        top,
        right,
        bottom
    ))

    zoomed = cropped.resize(
        (width, height),
        Image.Resampling.BILINEAR
    )

    return zoomed


def create_augmentations(image):
    """
    Create several simple label-preserving
    image augmentations.

    The exact transformations should always
    make sense for the chosen problem.
    """

    augmentations = {
        "hflip":
            ImageOps.mirror(image),

        "rot_p15":
            image.rotate(
                15,
                resample=Image.Resampling.BILINEAR,
                expand=False
            ),

        "rot_m15":
            image.rotate(
                -15,
                resample=Image.Resampling.BILINEAR,
                expand=False
            ),

        "bright_08":
            ImageEnhance.Brightness(
                image
            ).enhance(0.8),

        "bright_12":
            ImageEnhance.Brightness(
                image
            ).enhance(1.2),

        "contrast_12":
            ImageEnhance.Contrast(
                image
            ).enhance(1.2),

        "zoom_115":
            center_zoom(
                image,
                zoom_factor=1.15
            ),

        "blur":
            image.filter(
                ImageFilter.GaussianBlur(
                    radius=0.8
                )
            )
    }

    return augmentations


In [ ]:
# ============================================
# 19. AUGMENT THE DATASET
# ============================================

def augment_dataset(
    dataset_root,
    output_root,
    max_original_images=None
):
    """
    Create augmented copies while preserving
    the original folder/class structure.

    Example:
        original/class_A/img001.jpg

    becomes:
        augmented/class_A/
            img001__hflip.jpg
            img001__rot_p15.jpg
            ...
    """

    dataset_root = Path(
        dataset_root
    )

    output_root = Path(
        output_root
    )

    image_paths = find_images(
        dataset_root
    )

    if max_original_images is not None:
        image_paths = image_paths[
            :max_original_images
        ]

    if len(image_paths) == 0:

        print(
            "No images found. "
            "Augmentation skipped."
        )

        return 0

    augmented_count = 0

    for image_path in image_paths:

        relative_parent = (
            image_path.parent
            .relative_to(dataset_root)
        )

        output_directory = (
            output_root
            / relative_parent
        )

        output_directory.mkdir(
            parents=True,
            exist_ok=True
        )

        try:

            with Image.open(
                image_path
            ) as image:

                # Convert to RGB for consistent output
                image = image.convert("RGB")

                transformed = (
                    create_augmentations(
                        image
                    )
                )

                for (
                    augmentation_name,
                    augmented_image
                ) in transformed.items():

                    output_filename = (
                        f"{image_path.stem}"
                        f"__{augmentation_name}"
                        ".jpg"
                    )

                    output_path = (
                        output_directory
                        / output_filename
                    )

                    augmented_image.save(
                        output_path,
                        quality=95
                    )

                    augmented_count += 1

        except Exception as error:

            print(
                f"Skipped {image_path}: "
                f"{error}"
            )

    return augmented_count


In [ ]:
# Run on the full dataset after checking DATASET_ROOT.
#
# For a very quick demonstration first, set:
# max_original_images=10
#
# For the full lab task, use:
# max_original_images=None

augmented_count = augment_dataset(
    DATASET_ROOT,
    AUGMENTED_ROOT,
    max_original_images=None
)

print(
    "Number of augmented images created:",
    augmented_count
)


In [ ]:
# ============================================
# 20. PREVIEW ORIGINAL + AUGMENTATIONS
# ============================================

def preview_augmentations(
    image_path
):
    """
    Display one original image together with
    all augmentation variants.
    """

    with Image.open(
        image_path
    ) as image:

        image = image.convert("RGB")

        transformed = (
            create_augmentations(
                image
            )
        )

        images = {
            "original": image,
            **transformed
        }

        for name, current_image in images.items():

            plt.figure(figsize=(5, 4))

            plt.imshow(
                current_image
            )

            plt.title(name)
            plt.axis("off")
            plt.tight_layout()
            plt.show()


if len(original_image_paths) > 0:

    preview_augmentations(
        original_image_paths[0]
    )

else:

    print(
        "Preview skipped because no "
        "dataset images are available yet."
    )


In [ ]:
# ============================================
# 21. AUGMENTATION SUMMARY
# ============================================

augmented_image_paths = (
    find_images(
        AUGMENTED_ROOT
    )
)

summary = pd.DataFrame({
    "Dataset": [
        "Original",
        "Augmented copies"
    ],
    "Number of Images": [
        len(original_image_paths),
        len(augmented_image_paths)
    ]
})

print(summary.to_string(index=False))

print("\nImportant observations:")

print(
    "1. Augmentation creates new training "
    "examples by transforming existing images."
)

print(
    "2. A valid augmentation should preserve "
    "the correct class label."
)

print(
    "3. Keep original and augmented files "
    "clearly separated for reproducibility."
)

print(
    "4. In a train/validation/test experiment, "
    "augment only the training portion; do not "
    "create augmented copies of validation or "
    "test images."
)


In [ ]:
# ============================================
# FINAL SUMMARY
# ============================================

print("=" * 70)
print("LAB 03 SUMMARY")
print("=" * 70)

print("\nCLASSIFICATION")
print(
    "A classifier maps an input feature vector "
    "to a discrete class label."
)

print("\nLINEAR DECISION BOUNDARY")
print(
    "The perceptron uses sign(theta^T x). "
    "theta^T x = 0 is the decision boundary."
)

print("\nCONSTANT FEATURE / LIFTING")
print(
    "Adding x0 = 1 absorbs the bias/intercept "
    "into the parameter vector."
)

print("\nPERCEPTRON")
print(
    "A misclassified sample triggers "
    "theta <- theta + eta*y*x."
)

print("\nNON-LINEAR CLASSIFICATION")
print(
    "A non-linear feature map can make the "
    "classes linearly separable in a new "
    "feature space."
)

print("\nIMAGE AUGMENTATION")
print(
    "Label-preserving transformations can "
    "increase training-set diversity."
)

print("=" * 70)
